# Notebook 03 - Feature Engineering

## Objetivo

Consolidar los datos y agregar características relevantes.

## Entrada
- `data/interim/contract_clean.csv`
- `data/interim/personal_clean.csv`
- `data/interim/internet_clean.csv`
- `data/interim/phone_clean.csv`

## Salida
- `data/processed/data_processed.csv`

In [26]:
## Inicialización

import pandas as pd
from scipy.stats import chi2_contingency

In [27]:
# Cargar archivos de datos

interim_path = '../data/interim/' # Directorio para datos limpios
df_contract = pd.read_csv(interim_path+'contract_clean.csv', parse_dates=['begin_date'])
df_personal = pd.read_csv(interim_path+'personal_clean.csv')
df_internet = pd.read_csv(interim_path+'internet_clean.csv')
df_phone = pd.read_csv(interim_path+'phone_clean.csv')

In [28]:
# Establecer le fecha final, de corte, de los datos

final_date = pd.Timestamp(year=2020, month=2, day=1)

<div class="alert alert-block alert-info">
<p>
<b>Importante</b>
Las fechas de churn, (end_date), corresponden al periodo nov 2019 - ene 2020 pero para feb 2020 sospechosamente no hay un solo registro aunque sí hay begin_date con ese periodo. 
</p>
<p>Para el modelado se <b>simplificará</b> esta situación al tratar todos los registros con fecha de end_date de la misma forma: se crearán dos nuevas columnas
<ul>
<li><pre>churn: 0 si end_date = 'No'
       1 si end_date != 'No'</pre></li>
<li><pre>months_as_customer: meses entre el inicio de contrato y la fecha de corte si end_date = No 
                    meses entre el inicio de contrato y el fin del contrato si end_date != No</pre></li>
 </ul>
</p>
</div>

<div class="alert alert-block alert-info">
<p>
En un modelado adicional el manejo será diferente. Se pensará en predecir el churn para el siguiente mes. Esto implica:
<ol>
<li>Eliminar los registros con begin_date = Feb-01-2020</li>
<li>Crear nuevos conjuntos de datos tomando los cortes en Dic-01-2019 y Ene-01-2020. Calcular months_as_customer correspondiente</li>
<li>Usar los datos con corte en Nov como entrenamiento, corte en Dic como validación, y los datos con corte en Ene como prueba</li>
</ol>
</p>
</div>

### Consolidar datos

In [29]:
df = pd.merge(df_personal, df_contract, on = 'customer_id', how = 'outer')
df = pd.merge(df, df_internet, on = 'customer_id', how = 'outer')
df = pd.merge(df, df_phone, on = 'customer_id', how = 'outer')

In [30]:
# Validar consolidación

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   customer_id        7043 non-null   object        
 1   gender             7043 non-null   object        
 2   senior_citizen     7043 non-null   object        
 3   partner            7043 non-null   object        
 4   dependents         7043 non-null   object        
 5   begin_date         7043 non-null   datetime64[ns]
 6   end_date           7043 non-null   object        
 7   type               7043 non-null   object        
 8   paperless_billing  7043 non-null   object        
 9   payment_method     7043 non-null   object        
 10  monthly_charges    7043 non-null   float64       
 11  total_charges      7043 non-null   float64       
 12  internet_service   5517 non-null   object        
 13  online_security    5517 non-null   object        
 14  online_b

### Manejar ausentes tras consolidar

In [31]:
# Reglas de negocio aplicada. Si para un customer_id no hay registros de internet y phone es porque el usuario no contrató dicho servicio.

for column in list(df_internet.columns):
    df[column]=df[column].fillna('No internet service')
for column in list(df_phone.columns):
    df[column]=df[column].fillna('No phone service')

### Nuevas características

In [32]:
# Función para calcular antigüedad del contrato

def diff_month(d1, d2):
    return abs((d1.year - d2.year) * 12 + d1.month - d2.month)

In [33]:
# Agregar la antigüedad del contrato al dataframe

df['months_as_customer']= df.apply(lambda row: diff_month(final_date,row['begin_date']) if row['end_date']=='No'
                                    else diff_month(pd.Timestamp(row['end_date']),row['begin_date']), axis=1)

In [34]:
# Nueva característica: 'churn' indica si el contrato ha finalizado o no

df['churn'] = df['end_date'].apply(lambda x: 0 if x == 'No' else 1)

In [35]:
# Validar nuevas características

df[['begin_date','end_date','months_as_customer','churn']].head()

,begin_date,end_date,months_as_customer,churn
0,2019-05-01,No,9,0
1,2019-05-01,No,9,0
2,2019-09-01,2020-01-01 00:00:00,4,1
3,2018-12-01,2020-01-01 00:00:00,13,1
4,2019-09-01,2019-12-01 00:00:00,3,1


### Características a descartar 

Se eliminarán *customer_id* al ser simplemente un id. También se elminarán *begin_date* y *end_date* pues se reemplazaron en efecto por *churn* y *months_as_customer*

In [36]:
df.drop(['customer_id','begin_date','end_date'],axis = 1, inplace = True)

Se sospecha que hay una relación directa entre (*monthly_charges*, *months_as_customer*) y *total_charges*. Si se confirma la sospecha también es buena idea eliminar una de ellas, en este caso se opta por eliminar *total_charges*

In [37]:
df[['monthly_charges', 'months_as_customer','total_charges']].assign(
    charges_calc = df['months_as_customer'] * df['monthly_charges']).head(3)

,monthly_charges,months_as_customer,total_charges,charges_calc
0,65.6,9,593.30,590.4
1,59.9,9,542.40,539.1
2,73.9,4,280.85,295.6


La relación aparentemente es muy fuerte, buscaremos un estadístico que confirme esto.

In [38]:
# Se calcula la correlación entre la columna calculada y tota_charges
df[['total_charges']].assign(
    charges_calc = df['months_as_customer'] * df['monthly_charges']).corr(method='pearson')

,total_charges,charges_calc
total_charges,1.000000,0.999561
charges_calc,0.999561,1.000000


La correlación es efectivamente casi igual a 1. Se procede a eliminar *total_charges*.

In [39]:
df.drop(['total_charges'],axis = 1, inplace = True)

Interesa buscar otras variables candidatas a descartar. Una variable es candidata a descartar si es independiente de la variable objetivo. Dicho de otra forma, si para nuestra variable objetivo se observan las mismas frecuencias al filtrar por cada valor de la variable candidata. 

Sin embargo es posible que dos variables que parecen independientes sí tengan relación al agregar una tercera.
Se tomarán los candidatos tras hacer una prueba chi-cuadrada usando un valor de significancia $\alpha = 0.1$

In [40]:
# lista de variables excluyendo la respuesta (churn)

names = list(df.columns)
names.remove('churn')
alpha = 0.1
candidates = []
for column in names:
    tabla = pd.crosstab(df[column], df['churn'])
    chi2, p, _, _ = chi2_contingency(tabla)
    if p > alpha:
        candidates.append(column)
print(candidates)

['gender']


La variable **_gender_** es la única candidata a ser excluida del dataset. Una idea es comparar los resultados de los modelos con y sin esta variable. Otra idea es utilizar algoritmos que muestran la importancia de cada variable y en ese momento tomar la decisión antes de entrenar modelos más complejos. Se opta por este último enfoque.

In [41]:
df.drop(['gender'],axis = 1, inplace = True)

### Exportar datos

In [ ]:
proceseed_path = '../data/processed/' # Directorio para datos procesados
df.to_csv(proceseed_path+'dataset_processed.csv', index=False)